# 95 — SmolVLA tree Bellman fine-tuning, worker 0

Control arm: initialize from notebook 94's demonstration-pretrained Q10 critic, then train on matched 50/50 demonstration and v3-tree transition batches using an ordinary EMA Bellman HL-Gauss loss. Every tree transition has weight 1 and there is no fork-specific loss.


In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'

from google.colab import drive
drive.mount('/content/drive')

EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Use a Colab GPU runtime.'
DEMO_CACHE_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_demo_q10/cache')
DEMO_CHECKPOINT_PATH = Path('/content/drive/MyDrive/pnp_smolvla_demo_q10/checkpoints/*/demo_q10_pretrain/checkpoint_step_004000.pt')
TREE_CACHE_ROOT = Path('/content/smolvla_tree_bellman_cache')
OUTPUT_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_tree_bellman_finetune')
TREE_LIMIT = 480
UPDATES = 2000
BATCH_SIZE = 64
DOWNLOAD_WORKERS = 8
print({'arm': 'bellman_control', 'gpu': torch.cuda.get_device_name(0),
       'trees': TREE_LIMIT, 'updates': UPDATES, 'batch': BATCH_SIZE,
       'tree_weights': 'all 1', 'root_loss': 'off',
       'checkpoint_policy': 'latest only'})


In [ ]:
from pnp.smolvla_tree_bellman_finetune import run_smolvla_tree_bellman_finetune

report = run_smolvla_tree_bellman_finetune(
    run_name='bellman_control', fork_aware=False,
    demo_cache_root=DEMO_CACHE_ROOT,
    demo_checkpoint_path=DEMO_CHECKPOINT_PATH,
    tree_cache_root=TREE_CACHE_ROOT, output_root=OUTPUT_ROOT,
    tree_limit=TREE_LIMIT, updates=UPDATES, batch_size=BATCH_SIZE,
    download_workers=DOWNLOAD_WORKERS, resume=True)
report
